In [63]:
DATA = {
    "LAN": {
        "concept": "A Local Area Network connects computers within a small area like a building or campus.",
        "explanation": "LANs are privately owned networks used in offices, colleges, and homes. They are fast and reliable.",
        "example": "Computers connected in a college computer lab.",
        "follow_up": ["Want a comparison with WAN?", "Do you want a real-life example?"]
    },

    "WAN": {
        "concept": "A Wide Area Network connects computers over large geographical areas.",
        "explanation": "WANs connect networks across cities, countries, or continents using leased lines or satellites.",
        "example": "The Internet is the largest WAN.",
        "follow_up": ["Want WAN vs LAN comparison?", "Explain with a simple analogy?"]
    },

    "PAN": {
        "concept": "A Personal Area Network connects devices around a person.",
        "explanation": "PANs use short-range technologies like Bluetooth.",
        "example": "Bluetooth headphones connected to a phone.",
        "follow_up": ["Compare PAN with LAN?", "Want another daily example?"]
    },

    "MAN": {
        "concept": "A Metropolitan Area Network covers a city or large campus.",
        "explanation": "MANs connect multiple LANs within a metropolitan region.",
        "example": "City-wide cable TV or ISP network.",
        "follow_up": ["Want LAN vs MAN vs WAN comparison?", "Explain MAN using a city example?"]
    },

    "CLIENT-SERVER": {
        "concept": "In the client–server model, one machine provides services and others request them.",
        "explanation": "Servers store data or provide services. Clients request and use them over a network.",
        "example": "A web browser requesting a webpage from a web server.",
        "follow_up": ["Want peer-to-peer comparison?", "Explain with a simple diagram idea?"]
    },

    "PEER-TO-PEER": {
        "concept": "In peer-to-peer networks, all machines act as equals.",
        "explanation": "Each system can both request and provide services without a central server.",
        "example": "File sharing using BitTorrent.",
        "follow_up": ["Compare with client-server?", "Want a daily-life example?"]
    },

    "INTERNETWORK": {
        "concept": "An internetwork is a collection of interconnected networks.",
        "explanation": "Routers connect different networks to form an internetwork.",
        "example": "The Internet is a network of networks.",
        "follow_up": ["Internet vs WAN?", "Explain internetwork simply?"]
    }
}


In [80]:
INTENT_PATTERNS = {
    "GREETING": ["hi", "hello", "hey"],
    "ACK": ["ok", "okay", "good", "fine", "thanks", "thank you"],

    "DEFINE": ["what is", "define", "meaning", "means"],
    "EXPLAIN": ["explain", "tell me", "describe"],
    "EXAMPLE": ["example", "real life"],
    "COMPARE": ["compare", "difference", "vs"],

    "CONFUSED": [
        "didn't understand", "dont understand",
        "confused", "not clear", "again"
    ],

    "SUMMARY": ["summary", "summarize", "brief"],
}



In [82]:
def detect_intent(text):
    text = text.lower().strip()

    if text in ["exit", "quit", "bye"]:
        return "EXIT"

    if text.startswith("yes"):
        return "FOLLOW_UP_YES"

    if text in ["no", "not now", "skip"]:
        return "FOLLOW_UP_NO"

    for phrase in INTENT_PATTERNS["GREETING"]:
        if text == phrase:
            return "GREETING"

    for phrase in INTENT_PATTERNS["ACK"]:
        if text == phrase:
            return "ACK"

    for phrase in INTENT_PATTERNS["CONFUSED"]:
        if phrase in text:
            return "CONFUSED"

    for intent, phrases in INTENT_PATTERNS.items():
        if intent in ["CONFUSED", "GREETING", "ACK"]:
            continue
        for phrase in phrases:
            if phrase in text:
                return intent

    return "EXPLAIN"


In [64]:
def detect_topic(text):
    text = text.upper()

    for topic in DATA.keys():
        if topic in text:
            return topic

    # keyword mapping
    if "CLIENT" in text or "SERVER" in text:
        return "CLIENT-SERVER"
    if "PEER" in text:
        return "PEER-TO-PEER"
    if "INTERNET" in text:
        return "INTERNETWORK"

    return None


In [75]:
def generate_response(topic, intent):
    info = DATA[topic]

    if intent == "DEFINE":
        response = info["concept"]
        style = "definition"

    elif intent == "EXAMPLE":
        response = info["example"]
        style = "example"

    elif intent == "COMPARE":
        if topic == "PAN":
            response = (
                "PAN covers a few meters and connects personal devices. "
                "LAN covers a building or campus and connects many computers."
            )
        elif topic == "LAN":
            response = (
                "LAN is limited to a building and is very fast. "
                "WAN covers large distances and is slower."
            )
        elif topic == "WAN":
            response = (
                "WAN spans countries or continents. "
                "LAN is confined to a local area like an office."
            )
        else:
            response = "Please tell me what you want to compare."

        style = "comparison"

    elif intent == "CONFUSED":
        response = f"Think of {topic} like this: {info['example']}"
        style = "analogy"

    elif intent == "SUMMARY":
        response = info["concept"] + " " + info["explanation"]
        style = "summary"

    else:
        response = info["explanation"]
        style = "explanation"

    follow_up = info["follow_up"][0]

    return response, follow_up, style


In [60]:
from datetime import datetime

def log_interaction(topic, intent, style, follow_up):
    time = datetime.now().strftime("%Y-%m-%d %H:%M")
    with open("interaction_log.txt", "a") as f:
        f.write(f"{time} | {topic} | {intent} | {style} | {follow_up}\n")


In [85]:
print("Interactive Textbook Bot (type 'exit' to stop)\n")

last_topic = None
awaiting_followup = False
last_followup_intent = None

while True:
    user_input = input("You: ").strip()

    intent = detect_intent(user_input)
    topic = detect_topic(user_input)

    # ---- EXIT ----
    if intent == "EXIT":
        print("Bot: Bye!")
        break

    # ---- GREETING ----
    if intent == "GREETING":
        print("Bot: Hi! Ask me about LAN, WAN, or PAN.")
        continue

    # ---- ACKNOWLEDGEMENT ----
    if intent == "ACK":
        print("Bot: 👍 You can continue.")
        continue

    # ---- FOLLOW-UP NO ----
    if intent == "FOLLOW_UP_NO":
        print("Bot: Okay. Ask me anything else.")
        awaiting_followup = False
        last_followup_intent = None
        continue

   

    # ---- TOPIC RECOVERY ----
    if topic is None:
        if intent == "CONFUSED" and last_topic is not None:
            topic = last_topic
        else:
            print("Bot: Please mention a topic (LAN, WAN, PAN).")
            continue

    # ---- RESPONSE ----
    response, follow_up, style = generate_response(topic, intent)

    print("Bot:", response)
    print("Bot:", follow_up)

    # ---- STATE UPDATE ----
    last_topic = topic
    awaiting_followup = True

    # Infer follow-up intent from question asked
    if "compare" in follow_up.lower():
        last_followup_intent = "COMPARE"
    elif "example" in follow_up.lower():
        last_followup_intent = "EXAMPLE"
    else:
        last_followup_intent = None


Interactive Textbook Bot (type 'exit' to stop)



You:  hi


Bot: Hi! Ask me about LAN, WAN, or PAN.


You:  bye


Bot: Bye!
